In [1]:
# Включаем автоматическую перезагрузку внешних модулей
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
# Импортируем нашу функцию из созданного файла utils.py
import sys
import os

# Добавляем родительскую папку
sys.path.append(os.path.abspath('..'))

from src.improvements import (remove_zero_dimensions, 
                              encode_categories, 
                              evaluate_custom_model,
                              remove_outliers,
                              add_volume_feature)

In [2]:
df_raw = pd.read_csv('../data/raw/diamonds.csv')

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

base_lr = Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())])
robust_lr = Pipeline([('scaler', RobustScaler()), ('model', LinearRegression())])
base_rf = Pipeline([('scaler', StandardScaler()), ('model', RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1))])

In [3]:
df_raw = encode_categories(df_raw)

df_raw = remove_zero_dimensions(df_raw)
# Выбираем фичи и таргет без логарифма
X1 = df_raw[['carat', 'depth', 'table', 'x', 'y', 'z', 'cut_encoded', 'color_encoded', 'clarity_encoded']]
y1 = df_raw['price']

# Вызываем функцию
evaluate_custom_model(X1, y1, base_lr, is_target_logged=False)

📌 МОДЕЛЬ: Model (VAL STAGE)
R² (в долларах): 0.9130
CV R² (в долларах): 0.8875 (+/- 0.0355)
RMSE: 1174.76 $
MAE: 761.01 $
MAPE: 34.94%

Коэффициенты модели (после масштабирования):
        Признак  Коэффициент
          carat   +5172.2677
clarity_encoded    +818.8886
  color_encoded    +548.3445
    cut_encoded    +145.8018
              y    +131.9928
          depth     -27.4338
          table     -49.9803
              x    -398.3914
              z    -776.6396



In [4]:
# Применяем следующий шаг чистки
df_step2 = remove_outliers(df_raw)

X2 = df_step2[['carat', 'depth', 'table', 'x', 'y', 'z', 'cut_encoded', 'color_encoded', 'clarity_encoded']]
y2 = df_step2['price']
# Тестируем
evaluate_custom_model(X2, y2, base_lr, is_target_logged=False)

📌 МОДЕЛЬ: Model (VAL STAGE)
R² (в долларах): 0.9152
CV R² (в долларах): 0.9068 (+/- 0.0067)
RMSE: 1167.68 $
MAE: 760.33 $
MAPE: 34.33%

Коэффициенты модели (после масштабирования):
        Признак  Коэффициент
          carat   +5231.9661
              y   +2749.3974
clarity_encoded    +806.1225
  color_encoded    +545.0937
    cut_encoded    +138.0278
          depth     +84.9107
          table     -43.7843
              z   -1555.6394
              x   -2309.4605



In [5]:
df_step3 = add_volume_feature(df_step2)

X3 = df_step3[['carat', 'depth', 'table', 'volume', 'cut_encoded', 'color_encoded', 'clarity_encoded']]
y3 = df_step3['price']
# Тестируем
evaluate_custom_model(X3, y3, base_lr, is_target_logged=False)

📌 МОДЕЛЬ: Model (VAL STAGE)
R² (в долларах): 0.9140
CV R² (в долларах): 0.9044 (+/- 0.0053)
RMSE: 1175.83 $
MAE: 790.91 $
MAPE: 35.54%

Коэффициенты модели (после масштабирования):
        Признак  Коэффициент
         volume   +3116.3631
          carat   +1035.7856
clarity_encoded    +851.3286
  color_encoded    +536.1027
    cut_encoded    +117.8177
          depth      +3.8551
          table     -17.1433



In [6]:
X4 = df_step3[['carat', 'depth', 'table', 'volume', 'cut_encoded', 'color_encoded', 'clarity_encoded']]
y4 = np.log1p(df_step3['price']) # Берем логарифм от цены

print("=== МЕТРИКИ ПОСЛЕ ЛОГАРИФМИРОВАНИЯ ЦЕНЫ ===")
# Вызываем утилиту и передаем True, так как таргет залогорифмирован
evaluate_custom_model(X4, y4, base_lr, is_target_logged=True)

=== МЕТРИКИ ПОСЛЕ ЛОГАРИФМИРОВАНИЯ ЦЕНЫ ===
📌 МОДЕЛЬ: Model (VAL STAGE)
R² (в долларах): 0.8196
CV R² (в долларах): 0.8287 (+/- 0.0134)
RMSE: 0.43 $
MAE: 0.32 $
MAPE: 4.04%

Коэффициенты модели (после масштабирования):
        Признак  Коэффициент
         volume   +3116.3631
          carat   +1035.7856
clarity_encoded    +851.3286
  color_encoded    +536.1027
    cut_encoded    +117.8177
          depth      +3.8551
          table     -17.1433



In [ ]:
X5 = df_step3[['carat', 'depth', 'table', 'volume', 'cut_encoded', 'color_encoded', 'clarity_encoded']]

print("=== МЕТРИКИ ПОСЛЕ ЛОГАРИФМИРОВАНИЯ ЦЕНЫ ===")
# Вызываем утилиту и передаем True, так как таргет залогорифмирован.
evaluate_custom_model(X5, y4, base_lr, is_target_logged=True)

=== МЕТРИКИ ПОСЛЕ ЛОГАРИФМИРОВАНИЯ ЦЕНЫ ===
📌 МОДЕЛЬ: Model (VAL STAGE)
R² (в долларах): 0.8196
CV R² (в долларах): 0.8287 (+/- 0.0134)
RMSE: 0.43 $
MAE: 0.32 $
MAPE: 4.04%

Коэффициенты модели (после масштабирования):
        Признак  Коэффициент
         volume   +3116.3631
          carat   +1035.7856
clarity_encoded    +851.3286
  color_encoded    +536.1027
    cut_encoded    +117.8177
          depth      +3.8551
          table     -17.1433



In [8]:
# Изменяем список признаков: полностью убираем 'depth' и 'table'
X6 = df_step3[['carat', 'cut_encoded', 'color_encoded', 'clarity_encoded', 'volume']]
# Таргет оставляем залогорифмированным, так как он доказал свою эффективность для MAPE

print("=== МЕТРИКИ ПОСЛЕ УДАЛЕНИЯ DEPTH И TABLE ===")
evaluate_custom_model(X6, y4, base_lr, is_target_logged=True)

=== МЕТРИКИ ПОСЛЕ УДАЛЕНИЯ DEPTH И TABLE ===
📌 МОДЕЛЬ: Model (VAL STAGE)
R² (в долларах): 0.8191
CV R² (в долларах): 0.8282 (+/- 0.0135)
RMSE: 0.43 $
MAE: 0.32 $
MAPE: 4.05%



ValueError: All arrays must be of the same length

In [ ]:
# 1. Склеиваем огранку и цвет в один текстовый признак
df_step3['cut_color'] = df_step3['cut'].astype(str) + '_' + df_step3['color'].astype(str)

# 2. Быстро переводим этот новый признак в числа (кодируем через factorize)
df_step3['cut_color_encoded'] = pd.factorize(df_step3['cut_color'])[0]

# 3. Собираем новый список признаков (добавляем cut_color_encoded)
X7 = df_step3[['carat', 'cut_encoded', 'color_encoded', 'clarity_encoded', 'volume', 'cut_color_encoded']]

print("=== МЕТРИКИ ПОСЛЕ СКЛЕИВАНИЯ CUT + COLOR ===")
evaluate_custom_model(X7, y4, base_lr, is_target_logged=True)

=== МЕТРИКИ ПОСЛЕ СКЛЕИВАНИЯ CUT + COLOR ===
📌 МОДЕЛЬ: Model (VAL STAGE)
R² (в долларах): 0.8191
CV R² (в долларах): 0.8284 (+/- 0.0135)
RMSE: 0.43 $
MAE: 0.32 $
MAPE: 4.04%



ValueError: All arrays must be of the same length

In [ ]:
# 1. Бины для карата (проверяем округление до популярных коммерческих пиков)
# Округлим до 1 знака для надежности, чтобы поймать значения около пиков
df_step3['is_carat_pivot'] = df_step3['carat'].round(1).isin([0.3, 0.5, 0.7, 1.0, 1.5, 2.0]).astype(int)

# 2. Interaction term (Признак взаимодействия: Вес * Чистота)
df_step3['carat_clarity_inter'] = df_step3['carat'] * df_step3['clarity_encoded']

# 3. Склеивание Огранки и Чистоты (Cut + Clarity)
df_step3['cut_clarity'] = df_step3['cut'].astype(str) + '_' + df_step3['clarity'].astype(str)
df_step3['cut_clarity_encoded'] = pd.factorize(df_step3['cut_clarity'])[0]

# Собираем все наши новые и старые очищенные признаки вместе
X8 = df_step3[[
    'carat', 'volume', 'cut_encoded', 'color_encoded', 'clarity_encoded',
    'is_carat_pivot', 'carat_clarity_inter', 'cut_clarity_encoded'
]]

print("=== МЕТРИКИ С ПРОДВИНУТЫМ FEATURE ENGINEERING ===")
evaluate_custom_model(X8, y4, base_lr, is_target_logged=True)

=== МЕТРИКИ С ПРОДВИНУТЫМ FEATURE ENGINEERING ===
📌 МОДЕЛЬ: Model (VAL STAGE)
R² (в долларах): 0.8640
CV R² (в долларах): 0.8676 (+/- 0.0044)
RMSE: 0.37 $
MAE: 0.28 $
MAPE: 3.63%



ValueError: All arrays must be of the same length

In [ ]:
evaluate_custom_model(X8, y4, robust_lr, is_target_logged=True)

📌 МОДЕЛЬ: Model (VAL STAGE)
R² (в долларах): 0.8640
CV R² (в долларах): 0.8676 (+/- 0.0044)
RMSE: 0.37 $
MAE: 0.28 $
MAPE: 3.63%



In [ ]:
# Для Random Forest оставляем исходные размеры x/y/z: деревьям не мешает
# мультиколлинеарность, зато они используют больше геометрической информации.
X_rf = df_step3[[
    'carat', 'depth', 'table', 'x', 'y', 'z',
    'cut_encoded', 'color_encoded', 'clarity_encoded'
]]

evaluate_custom_model(X_rf, y4, base_rf, is_target_logged=True)

📌 МОДЕЛЬ: Model (VAL STAGE)
R² (в долларах): 0.9915
CV R² (в долларах): 0.9919 (+/- 0.0001)
RMSE: 0.09 $
MAE: 0.07 $
MAPE: 0.85%



In [14]:


from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Задаем базовый таргет (в долларах, без логарифма!)
y_final = df_step3['price']

# 1. Инициализируем новые пайплайны моделей
base_ridge = Pipeline([
    ('scaler', StandardScaler()), 
    ('model', Ridge(alpha=10.0)) # alpha — сила штрафа за мультиколлинеарность
])

base_gb = Pipeline([
    ('scaler', StandardScaler()), 
    ('model', GradientBoostingRegressor(n_estimators=100, random_state=42))
])

In [15]:
# Тестируем Ridge-регрессию
evaluate_custom_model(X8, y_final, base_ridge, is_target_logged=True, name="Ridge Regression")

# Тестируем стандартный Random Forest
evaluate_custom_model(X8, y_final, base_rf, is_target_logged=True, name="Random Forest")

# Тестируем Градиентный Бустинг
evaluate_custom_model(X8, y_final, base_gb, is_target_logged=True, name="Gradient Boosting")

📌 МОДЕЛЬ: Ridge Regression (VAL STAGE)
R² (в долларах): 0.8451
CV R² (в долларах): -3.7902 (+/- 3.9698)
RMSE: 1578.30 $
MAE: 887.15 $
MAPE: 23.52%

📌 МОДЕЛЬ: Random Forest (VAL STAGE)
R² (в долларах): 0.9787
CV R² (в долларах): 0.9790 (+/- 0.0009)
RMSE: 585.60 $
MAE: 301.92 $
MAPE: 8.63%

📌 МОДЕЛЬ: Gradient Boosting (VAL STAGE)
R² (в долларах): 0.9781
CV R² (в долларах): 0.9781 (+/- 0.0010)
RMSE: 593.75 $
MAE: 315.76 $
MAPE: 8.46%



In [17]:
from sklearn.model_selection import train_test_split, GridSearchCV # <-- Добавили импорт сюда
from sklearn.compose import TransformedTargetRegressor

# 1. Берем те 80% данных, которые выделены под обучение и валидацию
X_train_val, _, y_train_val, _ = train_test_split(X8, y_final, test_size=0.2, random_state=42)

# 2. Определяем сетку параметров, которые хотим примерить
param_grid = {
    'regressor__model__n_estimators': [50, 100],
    'regressor__model__max_depth': [15, 20, None]
}

# 3. Оборачиваем базовый rf_pipeline в TransformedTargetRegressor (авто-лог)
wrapped_rf = TransformedTargetRegressor(regressor=base_rf, func=np.log1p, inverse_func=np.expm1)

# 4. Настраиваем поиск
grid_search = GridSearchCV(wrapped_rf, param_grid, cv=3, scoring='r2', n_jobs=-1)
grid_search.fit(X_train_val, y_train_val)

print(f"Лучшие параметры: {grid_search.best_params_}")
best_tuned_model = grid_search.best_estimator_

Лучшие параметры: {'regressor__model__max_depth': 15, 'regressor__model__n_estimators': 100}


In [ ]:
print("🚀 ФИНАЛЬНЫЙ ЭКЗАМЕН МОДЕЛИ НА ТЕСТОВОЙ ВЫБОРКЕ 🚀")
evaluate_custom_model(
    X8, 
    df_step3['price'], 
    best_tuned_model, 
    is_target_logged=True, 
    name="Tuned Random Forest", 
    evaluation_stage='test'
)

🚀 ФИНАЛЬНЫЙ ЭКЗАМЕН МОДЕЛИ НА ТЕСТОВОЙ ВЫБОРКЕ 🚀
📌 МОДЕЛЬ: Tuned Random Forest (TEST STAGE)
R² (в долларах): 0.9813
CV R² (в долларах): 0.9801 (+/- 0.0009)
RMSE: 542.78 $
MAE: 284.30 $
MAPE: 8.05%

